In [ ]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

import pandas as pd
from datetime import datetime
import time


options = webdriver.ChromeOptions()
options.add_argument("--start-maximized")

driver = webdriver.Chrome(options=options)
wait = WebDriverWait(driver, 30)

try:
    
    url = "https://www.nseindia.com/market-data/live-equity-market?cat=EW50"

    print("Opening NSE...")
    driver.get(url)

    time.sleep(5)

    
    wait.until(
        EC.presence_of_element_located((By.TAG_NAME, "table"))
    )

    print("Table loaded...")

    
    tables = driver.find_elements(By.TAG_NAME, "table")

    target_table = None

    for table in tables:
        if "SYMBOL" in table.text and "LTP" in table.text:
            target_table = table
            break

    if target_table is None:
        raise Exception("NIFTY 50 table nahi mili.")

    # Extract rows
    rows = target_table.find_elements(By.TAG_NAME, "tr")

    data = []

    for row in rows:

        cells = row.find_elements(By.TAG_NAME, "th")

        if not cells:
            cells = row.find_elements(By.TAG_NAME, "td")

        row_data = [cell.text.strip() for cell in cells]

        if row_data:
            data.append(row_data)

    # Create DataFrame
    headers = data[0]

    valid_rows = [
        row for row in data[1:]
        if len(row) == len(headers)
    ]

    df = pd.DataFrame(
        valid_rows,
        columns=headers
    )

    # Remove duplicate stocks
    if "SYMBOL" in df.columns:
        df = df.drop_duplicates(subset=["SYMBOL"])

    # Date & time
    now = datetime.now()

    df["SCRAPED_DATE"] = now.strftime("%Y-%m-%d")
    df["SCRAPED_TIME"] = now.strftime("%H:%M:%S")

    # Check number of stocks
    print("\nStocks found:", len(df))

    # Save CSV
    date = now.strftime("%Y-%m-%d")

    csv_file = f"NIFTY50_{date}.csv"
    excel_file = f"NIFTY50_{date}.xlsx"

    df.to_csv(
        csv_file,
        index=False,
        encoding="utf-8-sig"
    )

    df.to_excel(
        excel_file,
        index=False
    )

    print("\nDONE ✅")
    print("CSV   :", csv_file)
    print("Excel :", excel_file)

    print("\nFirst 5 stocks:")
    print(df.head())

finally:
    driver.quit()

Opening NSE...
Table loaded...

Stocks found: 51

DONE ✅
CSV   : NIFTY50_2026-09-25.csv
Excel : NIFTY50_2026-09-25.xlsx

First 5 stocks:
      SYMBOL       OPEN       HIGH        LOW PREV.\nCLOSE        LTP  \
0   NIFTY 50  23,035.00  23,162.70  23,020.95    23,063.10  23,140.50   
1   RELIANCE   1,210.50   1,227.40   1,210.50     1,219.20   1,226.00   
2   HDFCBANK     723.00     739.65     723.00       728.90     735.60   
3  ICICIBANK   1,332.90   1,340.80   1,312.00     1,334.50   1,326.80   
4       INFY     999.50   1,004.20     991.60     1,014.50   1,000.20   

  INDICATIVE\nCLOSE  CHANGE % CHANGE VOLUME\n(shares) VALUE\n(₹ Crores)  \
0                 -   77.40     0.34     24,27,20,711         18,949.15   
1                 -    6.80     0.56      1,31,38,735          1,603.50   
2                 -    6.70     0.92      1,98,53,034          1,456.54   
3                 -   -7.70    -0.58      1,08,41,310          1,438.14   
4                 -  -14.30    -1.41      1,02,40